# Analyse de données immobilières avec Spark

Ce TP reprend, avec Spark, le travail du TP Seaborn, sur les mêmes données : le décompte des transactions par année, la valeur foncière par département, puis par région et par année, le prix au mètre carré et les corrélations. C'est voulu : à chaque exercice, comparez la durée des calculs (`%%time`, en première ligne d'une cellule, la mesure) et la mémoire occupée (la jauge RAM de Colab) avec celles de pandas. pandas chargeait en mémoire toutes les lignes des colonnes utiles ; Spark ne lit les données qu'au moment d'une action, et seulement les colonnes dont elle a besoin.

Deux exercices changent : la valeur par département se calcule avec l'API pandas de Spark, et la valeur par région et par année met en cache un résultat intermédiaire. S'y ajoutent la lecture d'un plan d'exécution et, en facultatif, l'écriture d'un Parquet partitionné. Une dernière partie, facultative, pour les formations qui ont vu la MLlib, entraîne un modèle de prix au mètre carré et le met en production : sauvegarde, prédictions par lots, puis en flux.

## Installation de PySpark

In [ ]:
!pip install pyspark

## Imports

In [ ]:
import pathlib
import time

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas
import pyspark
import pyspark.sql.functions as F
import seaborn as sns
from pyspark.sql import Column

## Chargement des données

Pour ces travaux pratiques, nous allons utiliser des données de transactions immobilières sur la France entière, entre 2014 et 2022.

Ces données proviennent du site d'[open data français](https://www.data.gouv.fr/fr/datasets/demandes-de-valeurs-foncieres-geolocalisees/). Nous utiliserons là une version retravaillée qui regroupe les transactions, qui provient du dépôt [NyxAether/DVF](https://github.com/NyxAether/DVF) sur GitHub.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh dvf

En Spark SQL, on commence toujours par définir une session, nommée par convention `spark`. Cette session permet de préciser comment accéder au cluster, quelques options, etc. Ici, nous l'utiliserons dans sa forme la plus simple.

In [ ]:
spark = pyspark.sql.SparkSession.builder.appName("DVF").getOrCreate()

Chargeons les données dans une feuille de données Spark, `df`. `dvf/dvf-linearized-2014-2022.parquet` est un dossier, partitionné par année : il contient un sous-dossier par valeur de `annee_mutation` (`annee_mutation=2014`, `annee_mutation=2015`…), d'où Spark refait cette colonne. `spark.read.parquet` n'y lit ici que la liste des fichiers et le schéma : le nom et le type de chaque colonne.

In [ ]:
df = spark.read.parquet("dvf/dvf-linearized-2014-2022.parquet")

Créez une colonne `date` à partir des colonnes `annee_mutation`, `mois_mutation` et `jour_mutation` avec `F.make_date` et `withColumn`, puis supprimez ces colonnes, que `date` remplace, avec `drop`. Affichez les dix premières lignes avec `show`.

**Documentation utile**

- [`pyspark.sql.functions.make_date`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.make_date.html)
- [`pyspark.sql.DataFrame.withColumn`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.withColumn.html)
- [`pyspark.sql.DataFrame.drop`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.drop.html)
- [`pyspark.sql.DataFrame.show`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.show.html)

In [ ]:
# Votre code ici

Enfin, affichez le type des colonnes de la feuille de données avec `printSchema`.

**Documentation utile**

- [`pyspark.sql.DataFrame.printSchema`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.printSchema.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`F.make_date` prend les noms des trois colonnes, dans l'ordre année, mois, jour. `withColumn` renvoie une nouvelle feuille de données, avec la colonne ajoutée, et `drop` une nouvelle feuille sans les colonnes nommées : affectez le résultat à `df`. `printSchema` affiche le nom et le type de chaque colonne : `date` doit y apparaître, en dernier, avec le type `date`.

```python
df = df.withColumn("date", F.make_date(...)).drop(...)
df.show(10)
df.printSchema()
```

### Solution

In [ ]:
df = df.withColumn(
  "date", F.make_date("annee_mutation", "mois_mutation", "jour_mutation")
).drop("annee_mutation", "mois_mutation", "jour_mutation")
df.show(10)

In [ ]:
df.printSchema()

## Évaluation paresseuse : le plan d'exécution

`withColumn` et `drop` n'ont rien calculé : ce sont des transformations, qui ajoutent une étape au plan de `df`. Seules les actions, comme `show`, `count` ou `toPandas`, lancent un calcul, et `show(10)` n'a lu que les premières lignes.

- Affichez le plan de `df` avec `explain()`. La ligne `FileScan parquet` donne, entre crochets, les colonnes que Spark lira dans le fichier : lesquelles ?
- Affichez de même le plan du décompte par année, `df.groupBy(F.year("date")).count()`, sans l'exécuter : quelles colonnes Spark lira-t-il cette fois ? Pourquoi peut-il ne lire qu'elles dans un fichier Parquet ?
- Comptez les lignes de `df` avec `count()` (vous devez trouver 10 839 063), puis réaffichez `df.explain()`. Le plan a-t-il changé ? Que refera Spark à la prochaine action sur `df` ?

**Documentation utile**

- [`pyspark.sql.DataFrame.explain`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.explain.html)
- [`pyspark.sql.DataFrame.count`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.count.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`df.explain()` affiche le plan physique, sans rien calculer. La ligne qui commence par `FileScan parquet` en est la lecture, et la liste entre crochets donne les colonnes lues : Spark en nomme 25, puis résume le reste (« ... 10 more fields »). Le plan du décompte est celui de la feuille de données que renvoie `count()` sur des données regroupées, une transformation : `df.groupBy(F.year("date")).count().explain()`. Sur la feuille entière, `count()` est en revanche une action, qui renvoie un entier.

```python
df.explain()
df.groupBy(...).count().explain()
print(df.count())
...
```

### Solution

In [ ]:
df.explain()
df.groupBy(F.year("date")).count().explain()
print(df.count())
df.explain()
# Le plan de df lit toutes les colonnes du fichier, qu'il garde toutes. Celui du
# décompte ne lit que jour_mutation, mois_mutation et annee_mutation, ce dont date a
# besoin : Parquet range chaque colonne à part, et annee_mutation vient du nom des
# sous-dossiers. count() a lu le fichier pour calculer un nombre, mais le plan de df
# n'a pas changé : Spark n'a rien gardé, et la prochaine action relira le fichier
# (cache(), plus loin, l'évite).

## Nombre de transactions par année

Pour commencer, nous allons créer le décompte par année du nombre de transactions (lignes du jeu de données).

Complétez la fonction `donnees_transactions_par_annee` : elle renvoie le nombre de transactions de chaque année, triées par année, dans une feuille de données pandas. Regroupez les lignes par année avec `groupBy` et `F.year`, appliqué à la colonne `date`, comptez-les avec `count`, triez-les avec `orderBy`, puis convertissez le résultat avec `toPandas`, qui renvoie une structure de données que seaborn sait exploiter. Décommentez ensuite l'appel `transactions_par_annee()`, en bas de la cellule, pour voir le diagramme.

La fonction de création de diagramme attend les données dans une feuille de données pandas organisée de cette manière :

```
   annee    count
0   2014  1032967
1   2015  1128516
2   2016  1210587
3   2017  1369101
4   2018  1351856
5   2019  1431358
6   2020  1268193
7   2021  1537648
8   2022   508837
```

**Documentation utile**

- [`pyspark.sql.DataFrame.groupBy`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupBy.html)
- [`pyspark.sql.functions.year`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.year.html)
- [`pyspark.sql.GroupedData.count`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.GroupedData.count.html)
- [`pyspark.sql.DataFrame.orderBy`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.orderBy.html)
- [`pyspark.sql.DataFrame.toPandas`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.toPandas.html)

In [ ]:
def donnees_transactions_par_annee() -> pandas.DataFrame:
  pass  # Votre code ici


def transactions_par_annee() -> None:
  donnees = donnees_transactions_par_annee()

  fig, ax = plt.subplots()
  sns.barplot(donnees, x="annee", y="count", ax=ax)
  sns.despine(fig)
  ax.set_title("Décompte des transactions par année")
  ax.set_xlabel("Année")
  ax.set_ylabel("Nombre de transactions")
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(places=1))
  fig.show()


# transactions_par_annee()

### Conseils supplémentaires

`F.year("date")` donne l'année de chaque date, et `.alias("annee")` nomme cette colonne, que le diagramme attend sous ce nom. `count()`, appelé sur des données regroupées, renvoie une feuille de données avec une colonne `count` : le nombre de lignes de chaque groupe. Spark ne garantit pas l'ordre des groupes : triez-les par `annee`.

```python
def donnees_transactions_par_annee() -> pandas.DataFrame:
  return df.groupBy(F.year("date").alias(...)).count().orderBy(...).toPandas()
```

### Solution

In [ ]:
def donnees_transactions_par_annee() -> pandas.DataFrame:
  return df.groupBy(F.year("date").alias("annee")).count().orderBy("annee").toPandas()


def transactions_par_annee() -> None:
  donnees = donnees_transactions_par_annee()

  fig, ax = plt.subplots()
  sns.barplot(donnees, x="annee", y="count", ax=ax)
  sns.despine(fig)
  ax.set_title("Décompte des transactions par année")
  ax.set_xlabel("Année")
  ax.set_ylabel("Nombre de transactions")
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(places=1))
  fig.show()


transactions_par_annee()

## Valeur foncière totale par département, avec l'API pandas

Dans cet exercice, nous allons regrouper nos données selon les valeurs d'une colonne (ici le département de la transaction) et afficher la valeur foncière totale échangée par groupe obtenu, cette fois avec l'API pandas de Spark : du code pandas, que Spark exécute.

Avec Spark 4.0, celui de Colab, l'API pandas refuse de démarrer en mode ANSI, que Spark 4 active par défaut : la cellule de l'exercice commence par le désactiver (c'est inutile depuis Spark 4.1).

Complétez la fonction `donnees_valeur_par_departement`, qui renvoie une série pandas :

- Passez à l'API pandas avec `pandas_api`, qui renvoie une feuille de données pandas-on-Spark.
- Regroupez-la par département (`code_departement`) et calculez la somme de `valeur_fonciere` de chaque groupe avec `groupby` et `sum`, puis triez les sommes par ordre décroissant avec `sort_values`, comme avec pandas.
- Convertissez le résultat en série pandas avec `to_pandas`.

Comparez ce code à celui du TP Seaborn pour le même diagramme. Décommentez ensuite l'appel `valeur_par_departement()` pour voir le diagramme.

La série pandas attendue a la forme suivante :

```
code_departement
75    1.821853e+11
92    1.327795e+11
6     8.143718e+10
33    8.015320e+10
69    7.632324e+10
          ...
55    1.998241e+09
15    1.907638e+09
52    1.753427e+09
23    1.248303e+09
48    8.985644e+08
Name: valeur_fonciere, Length: 97, dtype: float64
```

**Documentation utile**

- [`pyspark.sql.DataFrame.pandas_api`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.pandas_api.html)
- [`pyspark.pandas.DataFrame.groupby`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.pandas/api/pyspark.pandas.DataFrame.groupby.html)
- [`pyspark.pandas.Series.sort_values`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.pandas/api/pyspark.pandas.Series.sort_values.html)
- [`pyspark.pandas.Series.to_pandas`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.pandas/api/pyspark.pandas.Series.to_pandas.html)

In [ ]:
# Spark 4.0 (Colab) : l'API pandas refuse de démarrer en mode ANSI
spark.conf.set("spark.sql.ansi.enabled", "false")


def donnees_valeur_par_departement() -> pandas.Series:
  pass  # Votre code ici


def valeur_par_departement() -> None:
  donnees = donnees_valeur_par_departement()

  fig, ax = plt.subplots(figsize=(30, 10))
  sns.barplot(x=donnees.index, y=donnees, order=donnees.index, ax=ax)
  sns.despine(ax=ax)
  ax.set_title("Montant total des transactions par département")
  ax.set_xlabel("Département")
  ax.set_ylabel("Montant total des transactions")
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(unit="€", places=1))
  fig.show()


# valeur_par_departement()

### Conseils supplémentaires

`df.pandas_api()` renvoie une feuille de données pandas-on-Spark, dont les méthodes sont celles de pandas : `groupby` (en minuscules, comme dans pandas, et non `groupBy`), la sélection d'une colonne par son nom d'attribut, `sum`, `sort_values(ascending=False)`. Le résultat reste une série pandas-on-Spark, que Spark n'a pas encore calculée : `to_pandas()` lance le calcul et ramène les 97 sommes dans une série pandas.

```python
def donnees_valeur_par_departement() -> pandas.Series:
  df_pandas = df.pandas_api()
  return df_pandas.groupby(...).valeur_fonciere.sum().sort_values(...).to_pandas()
```

### Solution

In [ ]:
# Spark 4.0 (Colab) : l'API pandas refuse de démarrer en mode ANSI
spark.conf.set("spark.sql.ansi.enabled", "false")


def donnees_valeur_par_departement() -> pandas.Series:
  df_pandas = df.pandas_api()
  return (
    df_pandas.groupby("code_departement")
    .valeur_fonciere.sum()
    .sort_values(ascending=False)
    .to_pandas()
  )


def valeur_par_departement() -> None:
  donnees = donnees_valeur_par_departement()

  fig, ax = plt.subplots(figsize=(30, 10))
  sns.barplot(x=donnees.index, y=donnees, order=donnees.index, ax=ax)
  sns.despine(ax=ax)
  ax.set_title("Montant total des transactions par département")
  ax.set_xlabel("Département")
  ax.set_ylabel("Montant total des transactions")
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(unit="€", places=1))
  fig.show()


valeur_par_departement()

## Valeur foncière totale par région et par an

Nous allons étudier dans cette partie comment décliner une variable catégorielle en fonction d'une autre variable : nous allons produire un diagramme de la valeur foncière totale échangée par région, et cette valeur sera elle-même déclinée par année.

Pour cela, commençons par récupérer un jeu de données qui nous permettra de regrouper les départements en régions :

In [ ]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/france-departements/departements-region.csv -O regions.csv

La fonction suivante renvoie une feuille de données Spark qui permettra de joindre facilement la région à notre feuille de données de travail : sa colonne `num_dep` est le code du département, écrit comme dans nos données (« 1 », et non « 01 »), et sa colonne `region_name` le nom de la région.

In [ ]:
def lire_regions() -> pyspark.sql.DataFrame:
  return (
    spark.read.csv("regions.csv", inferSchema=True, header=True)
    .drop("dep_name")
    .withColumn("num_dep", F.regexp_replace("num_dep", "^0+", ""))
  )


lire_regions().show(10)

Complétez la fonction `donnees_valeur_par_region_par_annee`, qui renvoie une feuille de données et une série pandas :

- Ajoutez une colonne qui contient le nom de région à votre feuille de données de travail avec `join`, en faisant correspondre `code_departement` et `num_dep`, et ne gardez, avec `select`, que les trois colonnes utiles : la région, l'année de la date, sous le nom `annee`, et la valeur foncière.
- Cette feuille sert à deux agrégations, et Spark referait la lecture et la jointure pour chacune : mettez-la en cache avec `cache`, avant les deux.
- Renvoyez une feuille de données pandas qui contient la somme de la valeur foncière échangée par région et par année, sous le nom `valeur_totale`, avec `groupBy` sur deux colonnes.
- Renvoyez aussi une série pandas : les noms de région triés, de la région qui a la somme de valeur foncière échangée la plus importante à la moins importante.
- Mesurez la durée de chacune des deux agrégations avec `time.perf_counter`, puis libérez le cache avec `unpersist`.

Relancez ensuite la fonction sans `cache()`, et comparez les durées : quelle agrégation le cache accélère-t-il, laquelle ralentit-il, et pourquoi ? Quand vaut-il la peine ?

Décommentez ensuite l'appel `valeur_par_region_par_annee()` pour voir le diagramme.

La feuille de données à renvoyer a la forme suivante (l'ordre de ses lignes peut changer d'une exécution à l'autre) :

```
                 region_name  annee  valeur_totale
0              Île-de-France   2021   9.168523e+10
1    Bourgogne-Franche-Comté   2021   8.240557e+09
2                 La Réunion   2021   1.681132e+09
3                     Guyane   2021   5.290445e+08
4                   Bretagne   2021   1.613739e+10
..                       ...    ...            ...
148                Occitanie   2022   8.358505e+09
149               Guadeloupe   2022   5.992970e+07
150               La Réunion   2022   2.251319e+08
151                 Bretagne   2022   4.873580e+09
152                Normandie   2022   2.027011e+09
```

La série à renvoyer a la forme suivante :

```
0                  Île-de-France
1           Auvergne-Rhône-Alpes
2     Provence-Alpes-Côte d'Azur
3             Nouvelle-Aquitaine
4                      Occitanie
5                Hauts-de-France
6               Pays de la Loire
7                       Bretagne
8                      Normandie
9            Centre-Val de Loire
10       Bourgogne-Franche-Comté
11                     Grand Est
12                    La Réunion
13                         Corse
14                    Guadeloupe
15                    Martinique
16                        Guyane
Name: region_name, dtype: object
```

**Documentation utile**

- [`pyspark.sql.DataFrame.join`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html)
- [`pyspark.sql.DataFrame.select`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.select.html)
- [`pyspark.sql.DataFrame.cache`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.cache.html)
- [`pyspark.sql.DataFrame.unpersist`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.unpersist.html)
- [`time.perf_counter`](https://docs.python.org/fr/3/library/time.html#time.perf_counter)

In [ ]:
def donnees_valeur_par_region_par_annee() -> tuple[pandas.DataFrame, pandas.Series]:
  pass  # Votre code ici


def valeur_par_region_par_annee() -> None:
  donnees, ordre_regions = donnees_valeur_par_region_par_annee()

  fig, ax = plt.subplots(figsize=(20, 10))
  sns.barplot(
    donnees, x="region_name", y="valeur_totale", hue="annee", order=ordre_regions, ax=ax
  )
  sns.despine(fig)
  ax.set_title("Montant total des transactions par région et par année")
  ax.set_xlabel("Région")
  plt.xticks(rotation=60)
  ax.set_ylabel("Montant total des transactions")
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(unit="€", places=1))
  fig.show()


# valeur_par_region_par_annee()

### Conseils supplémentaires

`df.join(regions, df.code_departement == regions.num_dep)` garde les lignes dont le département a une région et leur ajoute les colonnes de `regions`. `select` reçoit des noms de colonnes ou des expressions, comme `F.year("date").alias("annee")`. `cache()` renvoie la même feuille de données, marquée pour être gardée en mémoire : Spark la calcule et la range à la première action qui s'en sert, puis la relit de là. Ne gardez que les trois colonnes : sans `select`, le cache garderait toutes celles du fichier. `groupBy` accepte plusieurs colonnes. Pour la série, regroupez par région seule, triez les sommes par ordre décroissant, gardez la colonne `region_name` avec `select`, puis prenez-la comme série après `toPandas()`, avec `.iloc[:, 0]`. `time.perf_counter()` renvoie un instant en secondes : la durée est la différence de deux appels. La fonction renvoie les deux résultats dans un tuple : `return donnees, ordre_regions`.

```python
def donnees_valeur_par_region_par_annee() -> tuple[pandas.DataFrame, pandas.Series]:
  regions = lire_regions()
  df_regions = (
    df.join(regions, ...).select("region_name", ..., "valeur_fonciere").cache()
  )
  debut = time.perf_counter()
  donnees = df_regions.groupBy(...).agg(...).toPandas()
  print(f"Par région et par année : {time.perf_counter() - debut:.1f} s")
  debut = time.perf_counter()
  ordre_regions = (
    df_regions.groupBy(...)
    .agg(...)
    .orderBy(..., ascending=False)
    .select("region_name")
    .toPandas()
    .iloc[:, 0]
  )
  print(f"Par région : {time.perf_counter() - debut:.1f} s")
  df_regions.unpersist()
  return donnees, ordre_regions
```

### Solution

In [ ]:
def donnees_valeur_par_region_par_annee() -> tuple[pandas.DataFrame, pandas.Series]:
  regions = lire_regions()
  df_regions = (
    df.join(regions, df.code_departement == regions.num_dep)
    .select("region_name", F.year("date").alias("annee"), "valeur_fonciere")
    .cache()
  )
  debut = time.perf_counter()
  donnees = (
    df_regions.groupBy("region_name", "annee")
    .agg(F.sum("valeur_fonciere").alias("valeur_totale"))
    .toPandas()
  )
  print(f"Par région et par année : {time.perf_counter() - debut:.1f} s")
  debut = time.perf_counter()
  ordre_regions = (
    df_regions.groupBy("region_name")
    .agg(F.sum("valeur_fonciere").alias("valeur_totale"))
    .orderBy("valeur_totale", ascending=False)
    .select("region_name")
    .toPandas()
    .iloc[:, 0]
  )
  print(f"Par région : {time.perf_counter() - debut:.1f} s")
  df_regions.unpersist()
  # La première agrégation calcule le cache et le range : elle est deux à trois fois
  # plus lente que sans cache. La seconde le relit : elle l'est deux fois moins. Au
  # total, le cache y perd ici, car relire trois colonnes de Parquet et joindre une
  # petite table coûte peu. Il paie quand une feuille sert souvent, ou coûte cher à
  # recalculer (une grosse jointure, un CSV à analyser).
  return donnees, ordre_regions


def valeur_par_region_par_annee() -> None:
  donnees, ordre_regions = donnees_valeur_par_region_par_annee()

  fig, ax = plt.subplots(figsize=(20, 10))
  sns.barplot(
    donnees, x="region_name", y="valeur_totale", hue="annee", order=ordre_regions, ax=ax
  )
  sns.despine(fig)
  ax.set_title("Montant total des transactions par région et par année")
  ax.set_xlabel("Région")
  plt.xticks(rotation=60)
  ax.set_ylabel("Montant total des transactions")
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(unit="€", places=1))
  fig.show()


valeur_par_region_par_annee()

## Prix des maisons et des appartements

Dans cet exercice, nous allons calculer le prix au mètre carré des appartements et maisons séparément et afficher l'évolution de ces prix dans l'intervalle considéré.

Complétez la fonction `donnees_prix_maisons_appartements` :

- Pour les appartements, filtrez la feuille de données de travail avec `filter` pour ne conserver que les lignes où :
    - `nombre_appartements`, `valeur_fonciere` et `surface_reelle_bati_appartements` sont strictement positifs ;
    - `nombre_maisons` vaut `0`.
- De la même manière, pour les maisons, filtrez la feuille de données de travail pour ne conserver que les lignes où :
    - `nombre_maisons`, `valeur_fonciere` et `surface_reelle_bati_maisons` sont strictement positifs ;
    - `nombre_appartements` vaut `0`.
- Utilisez ces données filtrées pour calculer le prix au mètre carré de chaque mois du jeu de données, pour les maisons et les appartements séparément : la somme des valeurs foncières du mois divisée par la somme des surfaces. `F.date_format` donne une colonne qui contient la date sous la forme `yyyy-MM`, par laquelle regrouper les lignes avec `groupBy`.
- Réunissez les prix des appartements et ceux des maisons avec `join`, sur le mois. La fonction de diagramme utilisée, `seaborn.lineplot`, attend des données en forme longue : `melt` les lui fournit.

Décommentez ensuite l'appel `prix_maisons_appartements()` pour voir le diagramme.

La forme de la feuille de données pandas attendue est la suivante :

```
        mois    type_bien         prix
0    2014-01  appartement  3005.224170
1    2014-01       maison  1588.628790
2    2014-02  appartement  2878.714589
3    2014-02       maison  1622.620962
4    2014-03  appartement  2911.870507
..       ...          ...          ...
199  2022-04       maison  2190.928353
200  2022-05  appartement  4164.208224
201  2022-05       maison  2150.579262
202  2022-06  appartement  4530.162208
203  2022-06       maison  2242.808145
```

**Documentation utile**

- [`pyspark.sql.DataFrame.filter`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.filter.html)
- [`pyspark.sql.functions.date_format`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.date_format.html)
- [`pyspark.sql.functions.col`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.col.html)
- [`pyspark.sql.DataFrame.melt`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.melt.html)
- [`seaborn.lineplot`](https://seaborn.pydata.org/generated/seaborn.lineplot.html)

In [ ]:
def donnees_prix_maisons_appartements() -> pandas.DataFrame:
  pass  # Votre code ici


def prix_maisons_appartements() -> None:
  donnees = donnees_prix_maisons_appartements()

  fig, ax = plt.subplots()
  sns.lineplot(donnees, x="mois", y="prix", hue="type_bien", legend=True)
  plt.xticks(rotation=45)
  ax.set_xticks(ax.get_xticks()[::12])
  ax.set_xlabel("Date")
  ax.set_ylabel("Valeur moyenne")
  ax.set_title("Prix au mètre carré")
  ax.set_ylim((0, donnees.select_dtypes("number").max().max()))
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(unit="€/m²", places=1))
  fig.show()


# prix_maisons_appartements()

### Conseils supplémentaires

Une condition sur une colonne, comme `df.nombre_appartements > 0`, est une `Column` de booléens, que `&` combine à une autre, chaque condition entre parenthèses. Une fonction intermédiaire, appelée une fois par type de bien, évite d'écrire deux fois le calcul : elle reçoit la condition, le nom de la colonne de surface et celui de la colonne de prix (`appartement` ou `maison`). Dans `agg`, nommez les deux sommes avec `.alias(...)` (`somme_valeur`, `somme_surface`) pour les diviser ensuite l'une par l'autre avec `F.col`. `join(autre, "mois")` réunit deux feuilles de données sur leur colonne commune, et `melt(ids="mois", values=["appartement", "maison"], variableColumnName="type_bien", valueColumnName="prix")` les met en forme longue, avant `toPandas()`.

```python
def donnees_prix_maisons_appartements() -> pandas.DataFrame:
  masque_appartements = (df.nombre_appartements > 0) & (df.nombre_maisons == 0) & ...
  masque_maisons = ...

  def prix_au_m2(
    masque: Column, colonne_surface: str, nom_colonne: str
  ) -> pyspark.sql.DataFrame:
    return (
      df.filter(masque)
      .groupBy(F.date_format("date", "yyyy-MM").alias("mois"))
      .agg(
        F.sum("valeur_fonciere").alias("somme_valeur"),
        F.sum(...).alias("somme_surface"),
      )
      .withColumn(nom_colonne, ...)
      .drop("somme_valeur", "somme_surface")
    )

  prix_appartements = prix_au_m2(
    masque_appartements, "surface_reelle_bati_appartements", "appartement"
  )
  prix_maisons = prix_au_m2(...)

  return (
    prix_appartements.join(prix_maisons, "mois").orderBy("mois").melt(...).toPandas()
  )
```

### Solution

In [ ]:
def donnees_prix_maisons_appartements() -> pandas.DataFrame:
  masque_appartements = (
    (df.nombre_appartements > 0)
    & (df.nombre_maisons == 0)
    & (df.valeur_fonciere > 0)
    & (df.surface_reelle_bati_appartements > 0)
  )
  masque_maisons = (
    (df.nombre_appartements == 0)
    & (df.nombre_maisons > 0)
    & (df.valeur_fonciere > 0)
    & (df.surface_reelle_bati_maisons > 0)
  )

  def prix_au_m2(
    masque: Column, colonne_surface: str, nom_colonne: str
  ) -> pyspark.sql.DataFrame:
    return (
      df.filter(masque)
      .groupBy(F.date_format("date", "yyyy-MM").alias("mois"))
      .agg(
        F.sum("valeur_fonciere").alias("somme_valeur"),
        F.sum(colonne_surface).alias("somme_surface"),
      )
      .withColumn(nom_colonne, F.col("somme_valeur") / F.col("somme_surface"))
      .drop("somme_valeur", "somme_surface")
    )

  prix_appartements = prix_au_m2(
    masque_appartements, "surface_reelle_bati_appartements", "appartement"
  )
  prix_maisons = prix_au_m2(masque_maisons, "surface_reelle_bati_maisons", "maison")

  return (
    prix_appartements.join(prix_maisons, "mois")
    .orderBy("mois")
    .melt(
      ids="mois",
      values=["appartement", "maison"],
      variableColumnName="type_bien",
      valueColumnName="prix",
    )
    .toPandas()
  )


def prix_maisons_appartements() -> None:
  donnees = donnees_prix_maisons_appartements()

  fig, ax = plt.subplots()
  sns.lineplot(donnees, x="mois", y="prix", hue="type_bien", legend=True)
  plt.xticks(rotation=45)
  ax.set_xticks(ax.get_xticks()[::12])
  ax.set_xlabel("Date")
  ax.set_ylabel("Valeur moyenne")
  ax.set_title("Prix au mètre carré")
  ax.set_ylim((0, donnees.select_dtypes("number").max().max()))
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(unit="€/m²", places=1))
  fig.show()


prix_maisons_appartements()

## Corrélations de Pearson et de Spearman

Comme au TP Seaborn, comparons les corrélations de Pearson et de Spearman, ici sur les seules ventes d'appartements de 2021, filtrées comme dans l'exercice précédent : au moins un appartement et aucune maison, une valeur foncière et une surface strictement positives. Les variables (la liste `variables`) sont des grandeurs : ni le code postal, ni le numéro de voie, ni les coordonnées, qui sont des identifiants.

- Gardez ces ventes et ces colonnes avec `filter` et `select`, passez à l'API pandas avec `pandas_api`, puis calculez les deux matrices de corrélation avec `corr` et son argument `method`.
- Affichez-les côte à côte, sur les deux `Axes` d'une même figure, avec `sns.heatmap`, chaque case annotée de sa valeur.
- Lisez la ligne de `valeur_fonciere` : avec `surface_reelle_bati_appartements`, vous devez trouver 0,35 (Pearson) et 0,41 (Spearman) ; avec `nombre_appartements`, 0,25 et 0,06.

Pourquoi le nombre d'appartements d'une vente est-il bien plus corrélé à sa valeur selon Pearson que selon Spearman ? Et pourquoi la surface n'explique-t-elle qu'une part de la valeur, même pour des appartements seuls ?

**Documentation utile**

- [`pyspark.pandas.DataFrame.corr`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.pandas/api/pyspark.pandas.DataFrame.corr.html)
- [`seaborn.heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html)

In [ ]:
variables = [
  "valeur_fonciere",
  "surface_reelle_bati_appartements",
  "nombre_pieces_principales",
  "nombre_appartements",
  "nombre_lots",
]


def correlations() -> None:
  pass  # Votre code ici


correlations()

### Conseils supplémentaires

La condition est celle des appartements de l'exercice précédent, avec en plus `F.year("date") == 2021`, et `select(variables)` garde les colonnes de la liste. Sur la feuille pandas-on-Spark (le mode ANSI est désactivé plus haut), `corr(method="spearman")` calcule la corrélation de Spearman (`"pearson"` par défaut) et renvoie une petite feuille pandas-on-Spark, que `to_pandas()` convertit pour seaborn. `fig, axes = plt.subplots(1, 2, figsize=(18, 7))` crée une figure de deux `Axes`, `axes[0]` et `axes[1]`, que `sns.heatmap` reçoit par `ax`. `annot=True` écrit la valeur dans chaque case, `fmt=".2f"` avec deux décimales ; `vmin=-1, vmax=1` fixent l'échelle des couleurs, et une palette divergente (`cmap="vlag"`) en distingue le signe.

```python
def correlations() -> None:
  appartements_2021 = (
    df.filter((F.year("date") == 2021) & (df.nombre_appartements > 0) & ...)
    .select(variables)
    .pandas_api()
  )
  fig, axes = plt.subplots(1, 2, figsize=(18, 7))
  for ax, methode in zip(axes, ["pearson", "spearman"]):
    sns.heatmap(
      appartements_2021.corr(method=...).to_pandas(),
      vmin=-1,
      vmax=1,
      cmap="vlag",
      annot=...,
      fmt=".2f",
      ax=ax,
    )
    ax.set_title(f"Corrélations ({methode}, appartements, 2021)")
  fig.tight_layout()
  fig.show()
```

### Solution

In [ ]:
def correlations() -> None:
  appartements_2021 = (
    df.filter(
      (F.year("date") == 2021)
      & (df.nombre_appartements > 0)
      & (df.nombre_maisons == 0)
      & (df.valeur_fonciere > 0)
      & (df.surface_reelle_bati_appartements > 0)
    )
    .select(variables)
    .pandas_api()
  )
  fig, axes = plt.subplots(1, 2, figsize=(18, 7))
  for ax, methode in zip(axes, ["pearson", "spearman"]):
    sns.heatmap(
      appartements_2021.corr(method=methode).to_pandas(),
      vmin=-1,
      vmax=1,
      cmap="vlag",
      annot=True,
      fmt=".2f",
      ax=ax,
    )
    ax.set_title(f"Corrélations ({methode}, appartements, 2021)")
  fig.tight_layout()
  fig.show()
  # 93 % de ces ventes ne portent que sur un appartement : Spearman, qui ne voit que
  # les rangs, ne lie presque pas leur nombre à la valeur (0,06). Pearson (0,25) suit
  # les quelques ventes en bloc, de plusieurs appartements et de grande valeur, qui
  # dominent la variance. La surface n'explique qu'une part de la valeur (0,35 et
  # 0,41), car le prix au mètre carré dépend du lieu : en 2021, sa médiane va de
  # 580 €/m² dans la Creuse à 10 700 €/m² à Paris, où Spearman monte à 0,83.


correlations()

## Écrire en Parquet partitionné par année (facultatif)

Le fichier DVF est partitionné par année : un sous-dossier par valeur de `annee_mutation`. Écrivons de même les ventes, réduites à quelques colonnes, et voyons ce que Spark y gagne à la lecture.

- Gardez de `df` les colonnes `date`, `code_departement` et `valeur_fonciere`, ajoutez-y une colonne `annee` (`F.year`), puis écrivez le tout en Parquet dans `ventes-par-annee.parquet`, partitionné par `annee`, avec `write`, `partitionBy` et `parquet` (`mode("overwrite")` permet de relancer la cellule).
- Listez le contenu du dossier écrit avec `pathlib.Path.iterdir`.
- Relisez-le avec `spark.read.parquet`, gardez les ventes de 2021 avec `filter`, et affichez leur plan avec `explain()` : que dit la ligne `PartitionFilters` ? Comptez-les : vous devez trouver 1 537 648 ventes.

**Documentation utile**

- [`pyspark.sql.DataFrame.write`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.write.html)
- [`pyspark.sql.DataFrameWriter.partitionBy`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html)
- [`pyspark.sql.DataFrameWriter.mode`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.mode.html)
- [`pyspark.sql.DataFrameWriter.parquet`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.parquet.html)
- [`pathlib.Path.iterdir`](https://docs.python.org/fr/3/library/pathlib.html#pathlib.Path.iterdir)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`df.write` renvoie un `DataFrameWriter`, dont les méthodes s'enchaînent : `partitionBy("annee")` donne la colonne qui nomme les sous-dossiers, `mode("overwrite")` remplace un dossier déjà écrit, et `parquet(chemin)` lance l'écriture (une action). `sorted(p.name for p in pathlib.Path(chemin).iterdir())` liste les sous-dossiers `annee=2014`… et le fichier `_SUCCESS`, qui marque une écriture terminée. Dans la feuille relue, `annee` est redevenue une colonne : un filtre sur elle apparaît dans `PartitionFilters`.

```python
(
  df.select(...)
  .withColumn("annee", ...)
  .write.partitionBy(...)
  .mode("overwrite")
  .parquet("ventes-par-annee.parquet")
)
print(sorted(p.name for p in pathlib.Path("ventes-par-annee.parquet").iterdir()))
ventes = spark.read.parquet(...)
ventes_2021 = ventes.filter(...)
ventes_2021.explain()
print(ventes_2021.count())
```

### Solution

In [ ]:
(
  df.select("date", "code_departement", "valeur_fonciere")
  .withColumn("annee", F.year("date"))
  .write.partitionBy("annee")
  .mode("overwrite")
  .parquet("ventes-par-annee.parquet")
)
print(sorted(p.name for p in pathlib.Path("ventes-par-annee.parquet").iterdir()))
ventes = spark.read.parquet("ventes-par-annee.parquet")
ventes_2021 = ventes.filter(ventes.annee == 2021)
ventes_2021.explain()
print(ventes_2021.count())
# PartitionFilters: [isnotnull(annee), (annee = 2021)] : Spark ne lit que le
# sous-dossier annee=2021, sans ouvrir ceux des autres années.

## Un modèle de prix au mètre carré (facultatif)

Cette partie, pour les formations qui ont vu la MLlib, passe de l'analyse à la prédiction : un modèle qui estime le prix au mètre carré d'un appartement d'après sa position, sa surface et son nombre de pièces, ajusté sur les ventes de 2021. Elle le suit jusqu'à sa mise en production : un `Pipeline` ajusté et évalué, sauvegardé, rechargé comme le ferait un autre programme, puis appliqué aux ventes de 2022, par lots, puis en flux.

Les dossiers que ces exercices écrivent (le modèle, les prédictions, le point de reprise du flux) sont relatifs au dossier courant, comme le Parquet partitionné plus haut, et chaque cellule les remplace : vous pouvez la relancer.

In [ ]:
import shutil

from pyspark.ml import Estimator, Pipeline, PipelineModel
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GBTRegressor, LinearRegression

### Les ventes d'appartements

Complétez la fonction `appartements`, qui prépare les ventes d'appartements d'une feuille de données de ventes, pour l'apprentissage comme pour les prédictions :

- filtrez-les comme dans la partie « Corrélations » : au moins un appartement et aucune maison, une valeur foncière et une surface strictement positives ;
- ajoutez-leur la cible, le prix au mètre carré, dans une colonne `prix_m2` ;
- gardez les prix compris entre 500 et 20 000 €/m², avec `between` : en dehors, ce sont surtout des cessions à prix symbolique, ou des ventes dont la valeur comprend autre chose que les appartements (2 % des ventes de 2021) ;
- gardez enfin la date, les colonnes de la liste `caracteristiques` et `prix_m2`.

Appliquez-la aux ventes de 2021, puis partagez le résultat en un ensemble d'apprentissage (80 %) et un ensemble de test (20 %) avec `randomSplit`. Les exercices suivants s'en servent plusieurs fois : mettez-les en cache, et comptez leurs lignes. Vous devez trouver environ 353 000 et 89 000 ventes.

**Documentation utile**

- [`pyspark.sql.Column.between`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Column.between.html)
- [`pyspark.sql.DataFrame.randomSplit`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.randomSplit.html)

In [ ]:
caracteristiques = [
  "longitude",
  "latitude",
  "surface_reelle_bati_appartements",
  "nombre_pieces_principales",
]


def appartements(ventes: pyspark.sql.DataFrame) -> pyspark.sql.DataFrame:
  pass  # Votre code ici


# Votre code ici

#### Conseils supplémentaires

La condition est celle des appartements de la partie « Corrélations », écrite sur `ventes` plutôt que sur `df`. `ventes.valeur_fonciere / ventes.surface_reelle_bati_appartements` divise deux colonnes ligne à ligne, et `F.col("prix_m2").between(500, 20000)` est vraie pour les prix compris entre les deux bornes, incluses. `select("date", *caracteristiques, "prix_m2")` déplie la liste en noms de colonnes. `randomSplit([0.8, 0.2], seed=0)` renvoie une liste de deux feuilles de données, à dépaqueter : `seed` rend le partage reproductible, et donne les nombres de l'énoncé. `cache()` renvoie la feuille marquée, que `count()` calcule et range.

```python
def appartements(ventes: pyspark.sql.DataFrame) -> pyspark.sql.DataFrame:
  return (
    ventes.filter((ventes.nombre_appartements > 0) & (ventes.nombre_maisons == 0) & ...)
    .withColumn("prix_m2", ...)
    .filter(...)
    .select(...)
  )


entrainement, test = appartements(df.filter(...)).randomSplit(...)
entrainement = entrainement.cache()
test = test.cache()
print(entrainement.count(), test.count())
```

#### Solution

In [ ]:
caracteristiques = [
  "longitude",
  "latitude",
  "surface_reelle_bati_appartements",
  "nombre_pieces_principales",
]


def appartements(ventes: pyspark.sql.DataFrame) -> pyspark.sql.DataFrame:
  return (
    ventes.filter(
      (ventes.nombre_appartements > 0)
      & (ventes.nombre_maisons == 0)
      & (ventes.valeur_fonciere > 0)
      & (ventes.surface_reelle_bati_appartements > 0)
    )
    .withColumn(
      "prix_m2", ventes.valeur_fonciere / ventes.surface_reelle_bati_appartements
    )
    .filter(F.col("prix_m2").between(500, 20000))
    .select("date", *caracteristiques, "prix_m2")
  )


entrainement, test = appartements(df.filter(F.year("date") == 2021)).randomSplit(
  [0.8, 0.2], seed=0
)
entrainement = entrainement.cache()
test = test.cache()
print(entrainement.count(), test.count())

### Un pipeline, deux modèles

Un `Pipeline` enchaîne des étapes : ici un `VectorAssembler`, qui réunit les colonnes de `caracteristiques` en un vecteur, dans la colonne `features` qu'attendent les modèles de la MLlib, puis un régresseur, qui prédit `prix_m2`. Son `fit` ajuste les étapes sur l'ensemble d'apprentissage et renvoie un `PipelineModel`, dont `transform` ajoute aux données la colonne `prediction`.

- Complétez la fonction `construire_pipeline`, qui renvoie le pipeline d'un régresseur donné.
- Complétez la fonction `evaluer`, qui affiche deux mesures des prédictions d'un modèle avec `RegressionEvaluator` : la racine de l'erreur quadratique moyenne (`rmse`, en €/m²) et le coefficient de détermination (`r2`).
- Ajustez le pipeline d'une `LinearRegression`, puis évaluez le modèle sur l'ensemble de test.
- Recommencez avec un `GBTRegressor` (des arbres de décision boostés) de 10 arbres, dans `modele`, et mesurez la durée de son `fit` avec `time.perf_counter`.

Vous devez trouver un R² proche de 0 pour la régression linéaire, et d'environ 0,68 pour les arbres, dont la RMSE est d'environ 1 600 €/m². Pourquoi un modèle linéaire n'explique-t-il presque rien du prix avec ces caractéristiques ? Qu'y gagnent les arbres ?

**Documentation utile**

- [`pyspark.ml.Pipeline`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.Pipeline.html)
- [`pyspark.ml.feature.VectorAssembler`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.VectorAssembler.html)
- [`pyspark.ml.regression.LinearRegression`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.regression.LinearRegression.html)
- [`pyspark.ml.regression.GBTRegressor`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.regression.GBTRegressor.html)
- [`pyspark.ml.evaluation.RegressionEvaluator`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.evaluation.RegressionEvaluator.html)

In [ ]:
def construire_pipeline(regresseur: Estimator) -> Pipeline:
  pass  # Votre code ici


def evaluer(modele: PipelineModel, donnees: pyspark.sql.DataFrame) -> None:
  pass  # Votre code ici


# Votre code ici

#### Conseils supplémentaires

`Pipeline(stages=[...])` reçoit la liste des étapes, dans l'ordre. `VectorAssembler(inputCols=caracteristiques, outputCol="features")` assemble les colonnes, et un régresseur nomme sa cible par `labelCol="prix_m2"` (sa colonne d'entrée est `features` par défaut). `GBTRegressor` prend le nombre d'arbres par `maxIter=10`, et `seed=0` rend l'ajustement reproductible. `RegressionEvaluator(labelCol="prix_m2", metricName="r2")` compare `prix_m2` à la colonne `prediction`, et son `evaluate` reçoit les données transformées par le modèle.

```python
def construire_pipeline(regresseur: Estimator) -> Pipeline:
  return Pipeline(stages=[VectorAssembler(...), regresseur])


def evaluer(modele: PipelineModel, donnees: pyspark.sql.DataFrame) -> None:
  predictions = modele.transform(donnees)
  for mesure in ["rmse", "r2"]:
    evaluateur = RegressionEvaluator(...)
    print(f"{mesure} : {evaluateur.evaluate(predictions):.2f}")


modele_lineaire = construire_pipeline(LinearRegression(...)).fit(entrainement)
evaluer(modele_lineaire, test)
debut = time.perf_counter()
modele = construire_pipeline(GBTRegressor(...)).fit(...)
print(f"Ajustement : {time.perf_counter() - debut:.0f} s")
evaluer(...)
```

#### Solution

In [ ]:
def construire_pipeline(regresseur: Estimator) -> Pipeline:
  return Pipeline(
    stages=[
      VectorAssembler(inputCols=caracteristiques, outputCol="features"),
      regresseur,
    ]
  )


def evaluer(modele: PipelineModel, donnees: pyspark.sql.DataFrame) -> None:
  predictions = modele.transform(donnees)
  for mesure in ["rmse", "r2"]:
    evaluateur = RegressionEvaluator(labelCol="prix_m2", metricName=mesure)
    print(f"{mesure} : {evaluateur.evaluate(predictions):.2f}")


modele_lineaire = construire_pipeline(LinearRegression(labelCol="prix_m2")).fit(
  entrainement
)
evaluer(modele_lineaire, test)
debut = time.perf_counter()
modele = construire_pipeline(GBTRegressor(labelCol="prix_m2", maxIter=10, seed=0)).fit(
  entrainement
)
print(f"Ajustement : {time.perf_counter() - debut:.0f} s")
evaluer(modele, test)
# Le prix au mètre carré dépend du lieu, mais pas en ligne droite : une fonction
# linéaire de la longitude et de la latitude ne décrit qu'une pente (plus cher au
# nord, ou à l'est), quand les prix culminent à Paris, sur la Côte d'Azur ou dans les
# Alpes. Les arbres découpent la carte en zones, chacune avec son prix, et croisent
# le lieu avec la surface.

### Sauvegarder et recharger le modèle

Un modèle sert ailleurs que là où il a été ajusté : dans un traitement par lots, chaque nuit, ou dans un service.

- Sauvegardez `modele` dans le dossier `modele-prix-m2` avec `write().overwrite().save(...)`, puis listez le contenu de son sous-dossier `stages` avec `pathlib.Path.iterdir` : qu'y trouvez-vous ?
- Rechargez-le dans `modele_recharge` avec `PipelineModel.load`, comme le ferait un autre programme, et vérifiez avec `evaluer` qu'il donne les mêmes mesures sur l'ensemble de test.

**Documentation utile**

- [`pyspark.ml.PipelineModel`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.PipelineModel.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`modele.write()` renvoie un écrivain, dont `overwrite()` permet de remplacer un dossier déjà écrit, et `save(chemin)` écrit le modèle. `PipelineModel.load(chemin)` est une méthode de la classe : elle n'a besoin d'aucun objet existant, seulement du dossier.

```python
modele.write().overwrite().save("modele-prix-m2")
print(sorted(p.name for p in pathlib.Path("modele-prix-m2/stages").iterdir()))
modele_recharge = PipelineModel.load(...)
evaluer(...)
```

#### Solution

In [ ]:
modele.write().overwrite().save("modele-prix-m2")
print(sorted(p.name for p in pathlib.Path("modele-prix-m2/stages").iterdir()))
modele_recharge = PipelineModel.load("modele-prix-m2")
evaluer(modele_recharge, test)
# stages contient un dossier par étape, numérotée dans l'ordre du pipeline : le
# VectorAssembler, avec ses colonnes, et le modèle ajusté, avec ses arbres. Le
# PipelineModel rechargé refait toute la préparation : le programme qui s'en sert n'a
# besoin que des colonnes brutes.

### Prédire par lots les ventes d'un mois de 2022

Un traitement par lots applique le modèle rechargé à un lot de données entier : ici les ventes d'appartements de juin 2022, le dernier mois des données, que le modèle, ajusté sur 2021, n'a jamais vues.

- Gardez de `df` les ventes de juin 2022 avec `F.year` et `F.month`, préparez-les avec `appartements`, puis appliquez-leur `modele_recharge.transform`.
- Écrivez les prédictions, sans la colonne `features`, dont leurs lecteurs n'ont pas besoin, en Parquet dans `predictions-2022-06.parquet` (`mode("overwrite")`).
- Relisez-les, comptez-les (vous devez trouver 25 181 ventes), et comparez la moyenne de `prix_m2` à celle de `prediction` avec `agg` et `F.avg`. Évaluez aussi le modèle sur ces ventes avec `evaluer` : sa RMSE passe d'environ 1 600 à environ 1 800 €/m².

Le modèle sous-estime-t-il ou surestime-t-il les prix de 2022 ? Pourquoi, et qu'en faire en production ? Ici, le vrai prix de chaque vente est connu, parce que DVF publie des ventes passées : en production, il ne l'est pas au moment de la prédiction.

**Documentation utile**

- [`pyspark.sql.functions.month`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.month.html)
- [`pyspark.sql.DataFrame.agg`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.agg.html)
- [`pyspark.sql.functions.avg`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.avg.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les deux conditions sur la date se combinent avec `&`, chacune entre parenthèses. `transform` renvoie une feuille de données, que Spark ne calcule qu'à l'écriture : `drop("features")`, puis `write.mode("overwrite").parquet(chemin)`, comme pour le Parquet partitionné. `agg(F.avg("prix_m2"), F.avg("prediction"))` calcule les deux moyennes en une passe, et `show()` les affiche.

```python
juin_2022 = appartements(df.filter((F.year("date") == 2022) & ...))
(
  modele_recharge.transform(juin_2022)
  .drop(...)
  .write.mode("overwrite")
  .parquet("predictions-2022-06.parquet")
)
predictions_juin = spark.read.parquet(...)
print(predictions_juin.count())
predictions_juin.agg(...).show()
evaluer(modele_recharge, juin_2022)
```

#### Solution

In [ ]:
juin_2022 = appartements(df.filter((F.year("date") == 2022) & (F.month("date") == 6)))
(
  modele_recharge.transform(juin_2022)
  .drop("features")
  .write.mode("overwrite")
  .parquet("predictions-2022-06.parquet")
)
predictions_juin = spark.read.parquet("predictions-2022-06.parquet")
print(predictions_juin.count())
predictions_juin.agg(F.avg("prix_m2"), F.avg("prediction")).show()
evaluer(modele_recharge, juin_2022)
# Le modèle sous-estime les prix de juin 2022 d'environ 5 % : les prix ont monté
# depuis 2021, et il ne connaît que ceux de 2021. C'est une dérive, que la
# surveillance d'un modèle en production détecte (la moyenne des prédictions, puis
# l'erreur une fois les vrais prix connus), et qu'un réajustement régulier sur les
# ventes récentes corrige.

### Prédire en flux (facultatif)

En flux, Spark traite les données à mesure qu'elles arrivent. Structured Streaming lit un dossier comme une table qui s'allonge, et ne traite à chaque passage que les fichiers nouveaux, grâce à son point de reprise (*checkpoint*), un dossier où il note ce qu'il a déjà traité.

La cellule de l'exercice simule l'arrivée des ventes : `deposer(mois)` écrit les ventes d'un mois de 2022 dans le dossier `arrivees`, partitionné par mois, et elle y dépose janvier et février. Le schéma de ces ventes est dans `schema_ventes`.

- Complétez la fonction `predire_en_flux` : lisez `arrivees` en flux avec `spark.readStream`, en lui donnant le schéma (un flux ne le devine pas), préparez les ventes avec `appartements`, appliquez-leur `modele_recharge.transform`, le même code que par lots, puis écrivez les prédictions, sans `features`, en Parquet dans `predictions-flux`, avec `writeStream`. Donnez-lui le point de reprise `predictions-flux-reprise` et `trigger(availableNow=True)`, qui traite tout ce qui est arrivé puis s'arrête, et attendez la fin avec `awaitTermination`.
- Appelez-la, puis comptez les prédictions écrites (`spark.read.parquet`). Déposez mars avec `deposer(3)`, rappelez-la et recomptez : qu'a lu le second passage ? Vous devez trouver 52 041, puis 82 293 prédictions.

**Documentation utile**

- [`pyspark.sql.SparkSession.readStream`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.readStream.html)
- [`pyspark.sql.streaming.DataStreamReader.schema`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.DataStreamReader.schema.html)
- [`pyspark.sql.streaming.DataStreamReader.parquet`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.DataStreamReader.parquet.html)
- [`pyspark.sql.DataFrame.writeStream`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.writeStream.html)
- [`pyspark.sql.streaming.DataStreamWriter.option`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.DataStreamWriter.option.html)
- [`pyspark.sql.streaming.DataStreamWriter.trigger`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.DataStreamWriter.trigger.html)
- [`pyspark.sql.streaming.DataStreamWriter.start`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.DataStreamWriter.start.html)
- [`pyspark.sql.streaming.StreamingQuery.awaitTermination`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.StreamingQuery.awaitTermination.html)

In [ ]:
# Repart de zéro à chaque exécution : ni ventes déposées, ni prédictions, ni reprise
for dossier in ["arrivees", "predictions-flux", "predictions-flux-reprise"]:
  shutil.rmtree(dossier, ignore_errors=True)
ventes_2022 = df.filter(F.year("date") == 2022).cache()


def deposer(mois: int) -> None:
  (
    ventes_2022.filter(F.month("date") == mois)
    .withColumn("mois", F.lit(mois))
    .write.partitionBy("mois")
    .mode("append")
    .parquet("arrivees")
  )


deposer(1)
deposer(2)
schema_ventes = spark.read.parquet("arrivees").schema


def predire_en_flux() -> None:
  pass  # Votre code ici


# Votre code ici

#### Conseils supplémentaires

`spark.readStream.schema(schema_ventes).parquet("arrivees")` renvoie une feuille de données en flux, à laquelle s'appliquent les mêmes transformations qu'à une autre. `writeStream` renvoie un `DataStreamWriter`, dont les méthodes s'enchaînent : `format("parquet")`, `option("checkpointLocation", chemin)`, `trigger(availableNow=True)`, puis `start(chemin)`, qui lance la requête et renvoie aussitôt une `StreamingQuery`. Son `awaitTermination()` attend qu'elle ait tout traité. Le second appel reprend du point de reprise : il ne lit que les fichiers de mars.

```python
def predire_en_flux() -> None:
  arrivees = spark.readStream.schema(...).parquet(...)
  requete = (
    modele_recharge.transform(appartements(arrivees))
    .drop(...)
    .writeStream.format("parquet")
    .option(...)
    .trigger(...)
    .start("predictions-flux")
  )
  requete.awaitTermination()


predire_en_flux()
print(spark.read.parquet("predictions-flux").count())
deposer(3)
...
```

#### Solution

In [ ]:
# Repart de zéro à chaque exécution : ni ventes déposées, ni prédictions, ni reprise
for dossier in ["arrivees", "predictions-flux", "predictions-flux-reprise"]:
  shutil.rmtree(dossier, ignore_errors=True)
ventes_2022 = df.filter(F.year("date") == 2022).cache()


def deposer(mois: int) -> None:
  (
    ventes_2022.filter(F.month("date") == mois)
    .withColumn("mois", F.lit(mois))
    .write.partitionBy("mois")
    .mode("append")
    .parquet("arrivees")
  )


deposer(1)
deposer(2)
schema_ventes = spark.read.parquet("arrivees").schema


def predire_en_flux() -> None:
  arrivees = spark.readStream.schema(schema_ventes).parquet("arrivees")
  requete = (
    modele_recharge.transform(appartements(arrivees))
    .drop("features")
    .writeStream.format("parquet")
    .option("checkpointLocation", "predictions-flux-reprise")
    .trigger(availableNow=True)
    .start("predictions-flux")
  )
  requete.awaitTermination()


predire_en_flux()
print(spark.read.parquet("predictions-flux").count())
deposer(3)
predire_en_flux()
print(spark.read.parquet("predictions-flux").count())
# Le second passage n'a lu que les fichiers de mars : le point de reprise garde la
# liste des fichiers déjà traités. Un traitement en flux réel tourne sans s'arrêter
# (un trigger toutes les minutes, par exemple), ou est relancé à intervalles réguliers
# avec availableNow, ce qui revient à un traitement par lots qui ne relit rien.